# ResNet-50: cuDNN-keuze, rekentijd en geheugen

**Korte synthetische diagnose · geen ImageNet-training**

We vergelijken twee werkelijk uitgevoerde controles met `cudnn.benchmark=True` met één nieuwe controle met `False`. Beide routes houden globale batch **256**, vier BN-groepen van **64**, alle vijf W/A-precisies, FP32 en dezelfde SGD-/loss-uitvoering. De twee bestaande controles worden niet opnieuw gestart.

De screenshots tonen bijna vol toegewezen GPU-geheugen en een kleine hoeveelheid gedeeld geheugen. Dat ondersteunt de hypothese van geheugendruk, maar stelt de volledige oorzaak van de vertraging niet vast. De waarde **15,6 GB** is de schaal van de grafiek en geen gemeten verbruik.


In [ ]:
from pathlib import Path
import importlib.util

RUN_DIRECTORY = ""
run_directory = Path(RUN_DIRECTORY).expanduser().resolve() if RUN_DIRECTORY else Path.cwd()
helper = run_directory / "notebook_support.py"
spec = importlib.util.spec_from_file_location("resnet50_backend_notebook_support", helper)
ns = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ns)
from IPython.display import display

run = ns.load_run(run_directory)
print("Resultatenmap:", run_directory)
print("Totale rapportstatus:", run["report"].get("status", "ONBEKEND"))
print(ns.outcome_text(run))

comparison_context = run["report"].get("comparison_context")
if comparison_context:
    print("Vergelijkingscontext:", comparison_context)


## 1. De volledige meetgegevens

Tijden zijn seconden per globale update, inclusief alle vijf precisies, forward, backward en SGD. Geheugen wordt weergegeven in GiB (2³⁰ bytes). De baselinewaarden komen rechtstreeks uit de twee aangeleverde feedbackbundels; runnamen blijven zichtbaar. Bij een onvoltooide proef kunnen ruwe deeltellers in de tabel staan. Die krijgen het meetbereik “Gedeeltelijk; piek afgebroken update onbekend”. De grafieken presenteren die niet als een geslaagde volledige meting. Een CPU-zelftest zonder GPU-proef krijgt GEEN_GPU_METING, ook als de CPU-test PASS was.


In [ ]:
table = ns.comparison_table(run)
display(table.round(3))


## 2. Staptijden na de opwarming

Iedere proef bevat één opwarmupdate en twee gemeten updates. De twee punten per proef blijven afzonderlijk zichtbaar; er wordt geen gemiddelde over onafhankelijke runs gemaakt.

De opwarming heeft uitgebreidere controle en synchronisatie. De gemeten updates starten ook met aangemaakte SGD-momentumstaat. Daarom vergelijken we de opwarmduur niet rechtstreeks met deze punten. Twee updates zijn nog geen stabiele schatting van de uiteindelijke trainingstijd.


In [ ]:
fig_times = ns.plot_step_times(run)
display(fig_times)
ns.plt.close(fig_times)


## 3. Geheugenpieken

`allocated` telt de ruimte voor PyTorch-tensors; `reserved` telt het geheugen dat de PyTorch-allocator beheert. De pieken gelden over de GPU-proef, inclusief opwarming. **Deze tellers meten niet rechtstreeks hoeveel geheugen fysiek in VRAM resident was.** Een waarde boven de gerapporteerde GPU-capaciteit bewijst op zichzelf geen Windows-geheugenfallback of de omvang daarvan.

De capaciteitslijn is uitsluitend context. Een OOM, TIMEOUT of ontbrekende nieuwe meting krijgt een statuslabel en geen nulstaaf.


In [ ]:
fig_memory = ns.plot_memory(run)
display(fig_memory)
ns.plt.close(fig_memory)


## 4. Interpretatie en vervolgbesluit

- **Duidelijk minder geheugen én kortere staptijden:** aanwijzing dat de cuDNN-keuze voor deze laptop relevant is. Daarna bepalen we of een echte pilot praktisch haalbaar is.
- **Vergelijkbare geheugendruk en snelheid:** deze ene backendwijziging lost de beperking niet op; een volgende maatregel vraagt een afzonderlijke proef.
- **OOM of TIMEOUT:** de nieuwe uitvoering is niet volledig gevalideerd. TIMEOUT bewijst geen OOM.

`benchmark=False` verandert de selectie van convolutionele kernels en mogelijk hun werkgeheugen. Het garandeert geen verbetering en geen bitexact gelijke CUDA-getallen. Model, batchgrootte, BN-groepen en wetenschappelijk trainingsrecept blijven behouden. **Activation checkpointing is niet ingevoerd.**

Er zijn nog geen ImageNet-accuracywaarden, volledige epochs of gevalideerde 80-epochprognoses in deze proef. Een PASS geeft geen automatische vrijgave voor ImageNet-training.


In [ ]:
test = run["report"].get("backend_test") or {}
for field in ("error", "last_progress", "elapsed_seconds", "timeout_seconds"):
    value = test.get(field)
    if value is not None:
        print(f"{field}: {value}")
if run["report"].get("error"):
    print("Rapportfout:", run["report"]["error"])


## Bronnen en herkomst

- Lokale meetdata: `baseline_results.json` en `run_report.json` in de gekozen resultatenmap. De baseline heeft twee gebruikersruns; de nieuwe gegevens staan in `backend_test`.
- [PyTorch 2.10 — cuDNN benchmarking](https://docs.pytorch.org/docs/2.10/notes/randomness.html#cuda-convolution-benchmarking).
- [PyTorch 2.10 — geheugenbeheer](https://docs.pytorch.org/docs/2.10/notes/cuda.html#memory-management).
- [NVIDIA — systeemgeheugenfallback](https://nvidia.custhelp.com/app/answers/detail/a_id/5490).
- [Microsoft — GPU-geheugen in Taakbeheer](https://devblogs.microsoft.com/directx/gpus-in-the-task-manager/).

Deze notebook voert alleen analyse uit; zij start geen training en wijzigt geen bestaande resultaten.
